# Tree Models - Training (CatBoost & XGBoost)

**Task 3: Tree-Based Model Architecture.** Turns the model configs into fitted, persisted
artifacts and verifies the `fit()` / `predict_proba()` / `save()` / `load()` contract from
`BaseCTRModel`. Runs on **Kaggle** (clones the repo) or **locally**; section 1 auto-detects which.

## Scope boundary

It **never reads the validation or test labels** - only feature matrices - so it cannot compute a
metric even by accident. Everything downstream belongs to other tasks in `CONTRIBUTING.md`:

| Belongs to | Task | Deliverable |
|---|---|---|
| ROC-AUC, LogLoss, PR-AUC, threshold reports | Task 4 | `src/evaluate/metrics.py`, `evaluator.py` |
| ROC / PR / Calibration curves | Task 4 | `src/evaluate/plot_results.py` -> `outputs/` |
| Hyperparameter search | Task 5 | `experiments/tune_optuna.py` |
| SHAP attributions, feature importance plots | Task 6 | `src/evaluate/shap_analysis.py` |

## What it produces

| Section | Output |
|---|---|
| 1 | Environment, data location, resolved paths |
| 2 | The two configs, side by side (feature scope + hyperparameters) |
| 3 | `models/catboost_fe.joblib` + `experiments/catboost_run.json` |
| 4 | `models/xgboost_fe.joblib` + `experiments/xgboost_run.json` |
| 5 | Interface check: save/load round-trip and `predict_proba` output contract |
| 6 | Handoff summary: which files to pass to Task 4 |

Equivalent on the command line, one model per command:

```bash
python -m src.models.run_catboost   # configs/catboost.yaml
python -m src.models.run_xgboost    # configs/xgboost.yaml
```

## 1. Environment setup

On Kaggle, `.gitignore` excludes `data/*`, `models/*`, `*.parquet` and `*.joblib`, so a clone
gives you code and configs only. Three things to arrange:

1. **Internet** - Settings -> Internet -> On, needed for the clone. Private repo: put a GitHub
   PAT in Add-ons -> Secrets under `GITHUB_TOKEN`.
2. **Data** - attach the engineered partitions (`train_fe.parquet`, `val_fe.parquet`,
   `test_fe.parquet`) as a Kaggle Dataset; the cell below searches `/kaggle/input` for them.
3. **Fit size** - `FIT_SAMPLE_SIZE` in section 2.

In [ ]:
import subprocess
import sys
from pathlib import Path

# ------------------------------------------------------------------ #
# Knobs
# ------------------------------------------------------------------ #
REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/CTR-prediction-on-Alibaba-Display-Advertising-Dataset.git"
BRANCH = "feat/models-catboost-xgboost"

# Private repo? Store a GitHub PAT in Kaggle Secrets under this name (Add-ons -> Secrets),
# and the clone URL becomes https://<token>@github.com/...
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"

ON_KAGGLE = Path("/kaggle/input").exists()
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'local'}")


def _run(cmd):
    """Run a shell command, surfacing failures instead of swallowing them."""
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True)


if ON_KAGGLE:
    repo_url = REPO_URL
    try:  # Optional: private-repo token from Kaggle Secrets
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
        repo_url = REPO_URL.replace("https://", f"https://{token}@")
        print("Using GitHub token from Kaggle Secrets.")
    except Exception:
        print("No GitHub token found - cloning as a public repo.")

    ROOT = Path("/kaggle/working") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if (ROOT / ".git").exists():
        print(f"Repo already cloned at {ROOT}, pulling latest.")
        _run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", BRANCH])
        _run(["git", "-C", str(ROOT), "reset", "--hard", f"origin/{BRANCH}"])
    else:
        _run(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, str(ROOT)])
else:
    # Local run: walk up from the notebook until the repo root is found.
    ROOT = Path.cwd()
    while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Repo root: {ROOT}")

In [ ]:
# Kaggle images ship polars / xgboost / catboost, but versions drift between image releases.
# Install only what is genuinely missing so this is a no-op on a warm environment.
import importlib.util

REQUIRED = {"polars": "polars", "xgboost": "xgboost", "catboost": "catboost", "yaml": "pyyaml"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f"Installing: {missing}")
    _run([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages already present.")

In [ ]:
import json
import logging
import warnings

import numpy as np
import polars as pl

from src.models.train import fit_from_config, load_config, load_dataset_from_config

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True)
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)


def find_processed_dir() -> Path:
    """
    Locate the directory holding the engineered parquet partitions.

    Local runs use the repo's own `data/processed`. On Kaggle that directory is empty (the data
    is gitignored), so every attached dataset under /kaggle/input is searched for train_fe.parquet.
    """
    local = ROOT / "data/processed"
    if (local / "train_fe.parquet").exists():
        return local

    if ON_KAGGLE:
        for candidate in sorted(Path("/kaggle/input").glob("**/train_fe.parquet")):
            return candidate.parent

    raise FileNotFoundError(
        "train_fe.parquet not found. On Kaggle, attach the engineered partitions as a Dataset "
        "(Add Data in the sidebar). Locally, run `python -m src.features.run_feature_engineering` first."
    )


# /kaggle/input is read-only; /kaggle/working (which holds the clone) is the writable location.
PROCESSED_DIR = find_processed_dir()
MODELS_DIR = ROOT / "models"
EXPERIMENTS_DIR = ROOT / "experiments"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processed data:  {PROCESSED_DIR}")
print(f"Model artifacts: {MODELS_DIR}")
print(f"Manifests:       {EXPERIMENTS_DIR}")
print("\nPartitions found:")
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    print(f"  {f.name:24s} {f.stat().st_size / 1e6:>8.1f} MB")

## 2. The two configs

One config per model, one runner per config, so a run never starts the other model.
`prepare_config` rewrites paths only - every hyperparameter and the whole feature scope come from
the YAML unchanged, so a fit here reproduces `python -m src.models.run_<model>`.

In [ ]:
# Rows to fit on. The engineered train partition is ~20M rows; a Kaggle kernel handles a few
# million comfortably. Set to 0 to use every row.
FIT_SAMPLE_SIZE = 1_000_000


def prepare_config(config_path: Path) -> dict:
    """
    Load a model config and rewrite its relative paths to absolute ones.

    The configs are written for `python -m src.models.run_<model>` executed from the repo root.
    A notebook has a different working directory (and on Kaggle the data lives outside the repo
    entirely), so paths are resolved once here and everything below just reads `cfg["paths"]`.
    """
    cfg = load_config(str(config_path))
    cfg["source_path"] = str(config_path)   # recorded in the manifest, for traceability
    cfg["paths"]["processed_dir"] = str(PROCESSED_DIR)
    cfg["paths"]["models_dir"] = str(MODELS_DIR)
    cfg["paths"]["manifest_output"] = str(
        EXPERIMENTS_DIR / Path(cfg["paths"]["manifest_output"]).name
    )
    return cfg


CATBOOST_CFG = prepare_config(ROOT / "configs/catboost.yaml")
XGBOOST_CFG = prepare_config(ROOT / "configs/xgboost.yaml")

# Feature scope declared by each config.
scope = pl.DataFrame([
    {
        "model": cfg["model"],
        "target": cfg["features"]["target"],
        "categorical": len(cfg["features"]["categorical"]),
        "numeric": len(cfg["features"]["numeric"]),
        "excluded": len(cfg["features"]["exclude_cols"]),
        "dropped": ", ".join(cfg["features"].get("drop_features") or []) or "-",
        "use_fe": cfg["data"]["use_fe"],
    }
    for cfg in (CATBOOST_CFG, XGBOOST_CFG)
])
scope

In [ ]:
# Hyperparameters, straight from each YAML.
for cfg in (CATBOOST_CFG, XGBOOST_CFG):
    print(f"--- {cfg['model']} ({cfg['source_path']}) ---")
    for k, v in cfg["params"].items():
        print(f"  {k:24s} {v}")
    print()

## 3. Fit CatBoost

`fit_from_config` is the function `python -m src.models.run_catboost` calls, so this cell and
that command produce the same artifact.

CatBoost keeps the raw advertiser IDs (`customer`, `brand`) - ordered target statistics already
regularize them. Early stopping tracks LogLoss, not AUC: AUC saturates in a few iterations and
would stop training before the probabilities are calibrated.

Roughly 5-15 minutes on 1M rows on a Kaggle CPU kernel; set `params.task_type: GPU` for a GPU one.

In [ ]:
catboost_run = fit_from_config(
    config=CATBOOST_CFG,
    config_path=CATBOOST_CFG["source_path"],
    model_key="catboost",
    sample_size=FIT_SAMPLE_SIZE,
)

print(f"\nArtifact: {catboost_run.artifact_path}")
print(f"Manifest: {catboost_run.manifest_path}")

## 4. Fit XGBoost

Independent of section 3 - it re-reads the partitions under its own config, so either section
runs alone.

Splitting on raw categorical levels let XGBoost memorize advertiser IDs: `customer` (255K levels),
`brand` (100K) and `campaign_id` (423K) took 41% of total gain, stopping at iteration 52/2000. Its
config drops `customer` and `brand` in favour of their target encodings; `campaign_id` has none
yet, so dropping it would remove the signal rather than regularize it.

In [ ]:
xgboost_run = fit_from_config(
    config=XGBOOST_CFG,
    config_path=XGBOOST_CFG["source_path"],
    model_key="xgboost",
    sample_size=FIT_SAMPLE_SIZE,
)

print(f"\nArtifact: {xgboost_run.artifact_path}")
print(f"Manifest: {xgboost_run.manifest_path}")

## 5. Interface check

`BaseCTRModel` promises that `save()` then `load()` restores a model predicting identically, and
that `predict_proba()` returns one probability per row in [0, 1]. Task 4 builds its evaluation
suite on those guarantees, so they are checked before handover.

This checks **our own interface**, not model quality: feature matrices only, `y_val` / `y_test`
never touched.

In [ ]:
from src.models.train import get_model_class

CHECK_ROWS = 5_000
checks = []

for run in (catboost_run, xgboost_run):
    key = run.model_key
    cfg = CATBOOST_CFG if key == "catboost" else XGBOOST_CFG

    # Features only - the labels stay untouched.
    X = run.dataset.X_val.head(CHECK_ROWS)

    reloaded = get_model_class(key).load(run.artifact_path)
    p_fitted = run.model.predict_proba(X)
    p_loaded = reloaded.predict_proba(X)

    checks.append({
        "model": key,
        "artifact_MB": round(Path(run.artifact_path).stat().st_size / 1e6, 2),
        "is_fitted": bool(reloaded.is_fitted),
        "features_match": reloaded.feature_names == run.model.feature_names,
        "roundtrip_identical": bool(np.allclose(p_fitted, p_loaded)),
        "shape_ok": p_loaded.shape == (len(X),),
        "in_0_1": bool(p_loaded.min() >= 0.0 and p_loaded.max() <= 1.0),
        "no_nan": bool(np.isfinite(p_loaded).all()),
        "p_min": round(float(p_loaded.min()), 5),
        "p_max": round(float(p_loaded.max()), 5),
    })

report = pl.DataFrame(checks)
report

In [ ]:
# Fail loudly rather than handing a broken artifact to Task 4.
CONTRACT = ["is_fitted", "features_match", "roundtrip_identical", "shape_ok", "in_0_1", "no_nan"]

for row in checks:
    failed = [c for c in CONTRACT if not row[c]]
    status = "OK" if not failed else f"FAILED -> {failed}"
    print(f"{row['model']:10s} {status}")

assert all(row[c] for row in checks for c in CONTRACT), "BaseCTRModel contract violated - do not hand off."
print("\nBoth artifacts satisfy the BaseCTRModel contract.")

## 6. Handoff

What Task 4 (`src/evaluate/`) and Task 5 (`experiments/tune_optuna.py`) consume. The manifest
records how each artifact was produced - sampling, seed, feature lists, dropped columns, best
iteration - so evaluation results stay traceable to a training run.

On Kaggle these land in `/kaggle/working`; save them as a Kaggle Dataset before the session ends,
since `.gitignore` keeps `*.joblib` out of git.

In [ ]:
print("Artifacts produced\n" + "=" * 70)
for run in (catboost_run, xgboost_run):
    art = Path(run.artifact_path)
    mf = Path(run.manifest_path) if run.manifest_path else None
    print(f"\n{run.model_key}")
    print(f"  model    {art}  ({art.stat().st_size / 1e6:.2f} MB)")
    print(f"  manifest {mf}")
    print(f"  fitted on {run.manifest['train_rows']:,} rows | "
          f"{run.manifest['n_features']} features | "
          f"best_iteration={run.manifest['best_iteration']} | "
          f"{run.manifest['train_seconds']}s")

print("\n\nHow Task 4 loads them\n" + "=" * 70)
print("""
from src.models.train import get_model_class, load_config, load_dataset_from_config

cfg = load_config("configs/catboost.yaml")
model = get_model_class("catboost").load("models/catboost_fe.joblib")

# Rebuild the same feature scope the model was fitted on
dataset = load_dataset_from_config(cfg, sample_size=200_000)
y_prob = model.predict_proba(dataset.X_test)

# Metrics and plots go in src/evaluate/ (Task 4), not here.
""".strip())